# Post-training 4/8-bit quantization of pruned VLMs

Loads a pruned LLaVA-v1.5-7B checkpoint (optionally recovered with distillation-LoRA), runs one decoding step end-to-end, and benchmarks single-token generation latency. Covers three configurations x two precisions:

| precision | unpruned | width-pruned 0.2 | width-pruned 0.4 |
|-----------|----------|------------------|------------------|
| FP16      | s1       | s3               | s4               |
| int8      | s2       | s5               | s6               |

Before running, set the environment variables defined in the next cell (or edit the defaults inline) so they point at your local checkpoints.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch
from PIL import Image

# --- Config (edit the first four paths or export them as env vars) ---
REPO_ROOT = Path(os.environ.get("REPO_ROOT", "..")).resolve()
IMAGE_PATH = os.environ.get(
    "QUANT_DEMO_IMAGE",
    str(REPO_ROOT / "VLM/llava/images/demo.jpg"),
)
# Pruned checkpoints produced by LLM-Pruner/examples/llava-vicuna_prune.py
WIDTH_0_2_CKPT = os.environ.get(
    "WIDTH_0_2_CKPT",
    str(REPO_ROOT / "LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.2_llava/pytorch_model.bin"),
)
WIDTH_0_4_CKPT = os.environ.get(
    "WIDTH_0_4_CKPT",
    str(REPO_ROOT / "LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.4_llava/pytorch_model.bin"),
)
# Distillation-recovered LoRA checkpoints paired with each pruned LM
WIDTH_0_2_LORA = os.environ.get(
    "WIDTH_0_2_LORA",
    str(REPO_ROOT / "VLM/llava/checkpoints/llava-v1.5-7b-0.2-dist-2.0-l2-0.5-layer--1"),
)
WIDTH_0_4_LORA = os.environ.get(
    "WIDTH_0_4_LORA",
    str(REPO_ROOT / "VLM/llava/checkpoints/llava-v1.5-7b-0.4-dist-2.0-l2-0.5-layer--1"),
)

sys.path.append(str(REPO_ROOT / "VLM"))
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import process_images

device = "cuda:0"
PROMPT = "Why is the image funny?"
SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user\'s questions."
)


def build_inputs(tokenizer, image_processor, model):
    text = f"{SYSTEM} USER: <image>\n{PROMPT} ASSISTANT:"
    chunks = [tokenizer(chunk).input_ids for chunk in text.split("<image>")]
    input_ids = torch.tensor(chunks[0] + [-200] + chunks[1], dtype=torch.long).unsqueeze(0).to(device)
    image = Image.open(IMAGE_PATH)
    image_tensor = process_images([image], image_processor, model.config).to(
        dtype=model.dtype, device=device
    )
    return input_ids, image_tensor


def measure(model, input_ids, image_tensor, max_new_tokens=15):
    model.to(device)
    start = time.time()
    out = model.generate(
        input_ids, images=image_tensor,
        max_new_tokens=max_new_tokens, use_cache=True,
    )[0]
    latency = time.time() - start
    print(f"Latency: {latency:.3f} s")
    print(f"GPU mem: {torch.cuda.memory_allocated() / 1024 / 1024:.1f} MiB")
    return tokenizer.decode(out, skip_special_tokens=True).strip()


## 1. Unpruned LLaVA-7b (FP16)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", device=device)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)


## 2. Unpruned LLaVA-7b (int8)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", load_8bit=True, device=device)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)


## 3. Width-pruned 0.2 (FP16)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all(
    "liuhaotian/llava-v1.5-7b", WIDTH_0_2_CKPT,
    lora=WIDTH_0_2_LORA, device=device)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)


## 4. Width-pruned 0.4 (FP16)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all(
    "liuhaotian/llava-v1.5-7b", WIDTH_0_4_CKPT,
    lora=WIDTH_0_4_LORA, device=device)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)


## 5. Width-pruned 0.2 (int8)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all(
    "liuhaotian/llava-v1.5-7b", WIDTH_0_2_CKPT,
    lora=WIDTH_0_2_LORA, device=device, load_8bit=True)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)


## 6. Width-pruned 0.4 (int8)

In [ ]:
tokenizer, model, image_processor, _ = load_pruned_llava_model_all(
    "liuhaotian/llava-v1.5-7b", WIDTH_0_4_CKPT,
    lora=WIDTH_0_4_LORA, device=device, load_8bit=True)
input_ids, image_tensor = build_inputs(tokenizer, image_processor, model)
print(measure(model, input_ids, image_tensor))


In [ ]:
%%timeit
model.generate(input_ids, images=image_tensor, max_new_tokens=1, use_cache=True)
